In [1]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain langchain-community
%pip install -Uq python_dotenv
%pip install -Uq sentence-transformers
%pip install -Uq supabase

You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart t

In [2]:
from supabase import create_client
from dotenv import load_dotenv
import os
load_dotenv()

SUPABASE_URL=os.getenv("SUPABASE_URL")
SUPABASE_KEY=os.getenv("SUPABASE_KEY")

supabase = create_client(
    SUPABASE_URL,
    SUPABASE_KEY
)

In [3]:
import json
from typing import List

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

# LangChain components
from langchain_core.documents import Document
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage
from langchain_community.vectorstores import SupabaseVectorStore


/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "./docs/constitution.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

The PDF <_io.BufferedReader name='./docs/constitution.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case


📄 Partitioning document: ./docs/constitution.pdf


The PDF <_io.BufferedReader name='./docs/constitution.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


✅ Extracted 3845 elements


In [5]:
elements

 ...]

In [6]:
# All types of different atomic elements we see from unstructured
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.Footer'>",
 "<class 'unstructured.documents.elements.Formula'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [7]:
len(elements)

3845

In [8]:
elements[0].to_dict()

{'type': 'Image',
 'element_id': '7c15ff0d9f542251c1dd189608f5f6cc',
 'text': '                                ',
 'metadata': {'coordinates': {'points': ((np.float64(668.6111111111111),
     np.float64(187.5)),
    (np.float64(668.6111111111111), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(187.5))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-05-13T23:42:41',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'constitution.pdf'}}

In [9]:
# Gather all images
images = [element for element in elements if element.category == 'Image']
print(f"Found {len(images)} images")

images[0].to_dict()

# Use https://codebeautify.org/base64-to-image-converter to view the base64 text

Found 15 images


{'type': 'Image',
 'element_id': '7c15ff0d9f542251c1dd189608f5f6cc',
 'text': '                                ',
 'metadata': {'coordinates': {'points': ((np.float64(668.6111111111111),
     np.float64(187.5)),
    (np.float64(668.6111111111111), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(562.5)),
    (np.float64(1036.9444444444443), np.float64(187.5))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-05-13T23:42:41',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './docs',
  'filename': 'constitution.pdf'}}

In [10]:
# Gather all table
tables = [element for element in elements if element.category == 'Table']
print(f"Found {len(tables)} tables")

tables[0].to_dict()

# Use https://jsfiddle.net/ to view the table html 

Found 28 tables


{'type': 'Table',
 'element_id': '78e327fe483387b18e5d258bb4de4da6',
 'text': '1. The Republic and its territories .................................................................... 3 2. Islam to be State religion ............................................................................... 3 2A. The Objectives Resolution to form part of substantive provisions ............ 3 3. Elimination of exploitation ............................................................................ 4 4. Right of individuals to be dealt with in accordance with law, etc. ............. 4 5. Loyalty to State and obedience to Constitution and law ............................. 4 6. High treason .................................................................................................... 4 7. Definition of the State..................................................................................... 6 8. Laws inconsistent with or in derogation of Fundamental Rights to be void .......................

In [11]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 245 chunks


In [12]:
# View all chunks
chunks

# All unique types
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>",
 "<class 'unstructured.documents.elements.TableChunk'>"}

In [13]:
# View a single chunk
# chunks[5].to_dict()

# View original elements
chunks[11].metadata.orig_elements[-1].to_dict()

{'type': 'UncategorizedText',
 'element_id': 'fc257f981fcc5b2bcb765acc2571ab68',
 'text': 'CHAPTER 2. – THE SUPREME COURT OF PAKISTAN ........................................... 96',
 'metadata': {'coordinates': {'points': ((np.float64(350.0555555555555),
     np.float64(493.7222222222219)),
    (np.float64(350.0555555555555), np.float64(525.2158888888886)),
    (np.float64(1355.875111111111), np.float64(525.2158888888886)),
    (np.float64(1355.875111111111), np.float64(493.7222222222219))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-05-13T23:42:41',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 8,
  'parent_id': 'b2f8e16ad1d13838a9513cf61cde14e0',
  'file_directory': './docs',
  'filename': 'constitution.pdf'}}

In [14]:
# here we have a chunk with text and a table. We want to be able to identify both the text and the table(html), and store them in a structured way so we can use them later for retrieval and question answering.

def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data



In [15]:
def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content"""
    
    try:
        # Initialize LLM (needs vision model for images)
        llm = ChatOllama(model="mistral",temperature=0)
        
        # Build the text prompt
        prompt_text = f"""You are creating a searchable description for document content retrieval.

        CONTENT TO ANALYZE:
        TEXT CONTENT:
        {text}

        """
        
        # Add tables if present
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
                prompt_text += """
                YOUR TASK:
                Generate a comprehensive, searchable description that covers:

                1. Key facts, numbers, and data points from text and tables
                2. Main topics and concepts discussed  
                3. Questions this content could answer
                4. Visual content analysis (charts, diagrams, patterns in images)
                5. Alternative search terms users might use

                Make it detailed and searchable - prioritize findability over brevity.

                SEARCHABLE DESCRIPTION:"""

        # Build message content starting with text
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Add images to the message
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        # Send to AI and get response
        message = HumanMessage(content=message_content)
        response = llm.invoke([message])
        
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        # Fallback to simple summary
        summary = f"{text[:300]}..."
        if tables:
            summary += f" [Contains {len(tables)} table(s)]"
        if images:
            summary += f" [Contains {len(images)} image(s)]"
        return summary

In [16]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    langchain_documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

In [17]:
# Process chunks with AI
processed_chunks = summarise_chunks(chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/245
     Types found: ['text', 'image']
     Tables: 0, Images: 1
     → Creating AI summary for mixed content...
     ❌ AI summary failed: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))
     → AI summary created successfully
     → Enhanced content preview: THE CONSTITUTION OF THE ISLAMIC REPUBLIC OF PAKISTAN

[As modified upto the 28th February, 2012]

NATIONAL ASSEMBLY OF PAKISTAN

PREFACE

The National Assembly of Pakistan passed the Constitution on 1...
   Processing chunk 2/245
     Types found: ['text']
     Tables: 0, Images: 0
     → Using raw text (no tables/images)
   Processing chunk 3/245
     Types found: ['text', 'table']
     Tables: 1, Images: 0
     → Creating AI summary for mixed content...
     ❌ AI summary failed: 

/var/folders/wf/vp93xvkj64v8swj3nrn337yr0000gn/T/ipykernel_43230/149721099.py:6: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="mistral",temperature=0)


In [18]:
processed_chunks

[Document(metadata={'original_content': '{"raw_text": "THE CONSTITUTION OF THE ISLAMIC REPUBLIC OF PAKISTAN\\n\\n[As modified upto the 28th February, 2012]\\n\\nNATIONAL ASSEMBLY OF PAKISTAN\\n\\nPREFACE\\n\\nThe National Assembly of Pakistan passed the Constitution on 10th April, 1973, the President of the Assembly authenticated it on 12th April, 1973 and the Assembly published the Constitution of the Islamic Republic of Pakistan. Since then, a number of amendments have been made therein and it has become necessary and expedient that an up-to-date and authentic version of the Constitution be published by the Assembly.\\n\\nThe present Sixth Edition of the Constitution is distinctive as it contains three historic amendments:\\n\\n(i) The Constitution (Eighteenth Amendment) Act, 2010 (Act X of 2010), adopted by the National Assembly on the 8th April, 2010, and by the Senate of Pakistan on the 15th April, 2010, unanimously, and assented to by the President on the 19th April, 2010;\\n\\n(

In [19]:
def export_chunks_to_json(chunks, filename="chunks_export.json"):
    """Export processed chunks to clean JSON format"""
    export_data = []
    
    for i, doc in enumerate(chunks):
        chunk_data = {
            "chunk_id": i + 1,
            "enhanced_content": doc.page_content,
            "metadata": {
                "original_content": json.loads(doc.metadata.get("original_content", "{}"))
            }
        }
        export_data.append(chunk_data)
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Exported {len(export_data)} chunks to {filename}")
    return export_data

# Export your chunks
json_data = export_chunks_to_json(processed_chunks)

✅ Exported 245 chunks to chunks_export.json


In [20]:
def create_vector_store(documents):
    print("🔮 Creating embeddings and storing in pgvector...")

    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    vectorstore = SupabaseVectorStore.from_documents(
        documents=documents,
        embedding=embedding_model,
        client=supabase,
        table_name="documents",
        query_name="match_documents"
    )

    print("✅ Vector store created")

    return vectorstore

db = create_vector_store(processed_chunks)

🔮 Creating embeddings and storing in pgvector...


/var/folders/wf/vp93xvkj64v8swj3nrn337yr0000gn/T/ipykernel_43230/2458531978.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


✅ Vector store created


In [38]:
def retrieve_from_supabase(query: str, k: int = 3):
    """Direct Supabase RPC retrieval — bypasses SupabaseVectorStore which is broken with supabase>=2.0"""
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    query_embedding = embedding_model.embed_query(query)

    result = supabase.rpc(
        "match_documents",
        {
            "query_embedding": query_embedding,
            "match_count": k,
            "filter": {},
        }
    ).execute()

    return [
        Document(
            page_content=row["content"],
            metadata=row.get("metadata", {})
        )
        for row in result.data
    ]

query = "What is 18th amendment?"
chunks = retrieve_from_supabase(query, k=5)

export_chunks_to_json(chunks, "rag_results.json")

✅ Exported 5 chunks to rag_results.json


[{'chunk_id': 1,
  'enhanced_content': 'PART XI\n\nAmendment of Constitution\n\n238. Amendment of Constitution\n\n238. Subject to this Part, the Constitution may be amended by Act of 1[Majlis-e-Shoora (Parliament)].\n\n239. Constitution, amendment Bill\n\n2[239. (1) A Bill to amend the Constitution may originate in either House and, when the Bill has been passed by the votes of not less than two-thirds of the total membership of the House, it shall be transmitted to the other House.\n\n(2) If the Bill is passed without amendment by the votes of not less than two-thirds of the total membership of the House to which it is transmitted under clause (1), it shall, subject to the provisions of clause (4), be presented to the President for assent.\n\n(3) If the Bill is passed with amendment by the votes of not less than two-thirds of the total membership of the House to which it is transmitted under clause (1), it shall be reconsidered by the House in which it had originated, and if the Bill 

In [41]:
def run_complete_ingestion_pipeline(pdf_path: str):
    """Run the complete RAG ingestion pipeline"""
    print("🚀 Starting RAG Ingestion Pipeline")
    print("=" * 50)
    
    # Step 1: Partition
    elements = partition_document(pdf_path)
    
    # Step 2: Chunk
    chunks = create_chunks_by_title(elements)
    
    # Step 3: AI Summarisation
    summarised_chunks = summarise_chunks(chunks)
    
    # Step 4: Vector Store
    db = create_vector_store(summarised_chunks)
    
    print("🎉 Pipeline completed successfully!")
    return db

In [ ]:
# Run the complete pipeline
db = run_complete_ingestion_pipeline("./docs/constitution.pdf")

The PDF <_io.BufferedReader name='./docs/constitution.pdf'> contains a metadata field indicating that it should not allow text extraction. Ignoring this field and proceeding. Use the check_extractable if you want to raise an error in this case


🚀 Starting RAG Ingestion Pipeline
📄 Partitioning document: ./docs/constitution.pdf


# RETREIVAL PIPELINE

In [ ]:
%pip install langchain_cohere


     |████████████████████████████████| 42 kB 332 kB/s eta 0:00:01
     |████████████████████████████████| 334 kB 742 kB/s eta 0:00:01
     |████████████████████████████████| 980 kB 6.6 MB/s eta 0:00:01
You should consider upgrading via the '/Users/anus/Workspace/Anus/AI/Constitution-RAG/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [28]:
from langchain.retrievers import EnsembleRetriever, BM25Retriever
from langchain_cohere import CohereRerank

#### SETUP: Create the three types of retrievers

# 1. Vector Retriever (Semantic Search/Dense Retrieval)
Setting up hybrid retriever...

In [ ]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun

class SupabaseRetriever(BaseRetriever):
    """Custom retriever using direct RPC — compatible with supabase>=2.0"""
    k: int = 15

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun):
        return retrieve_from_supabase(query, k=self.k)

vector_retriever = SupabaseRetriever(k=15)

# 2. BM25 Retriever (Keyword Search/Sparse Retrieval)

In [29]:
# 2. BM25 Retriever
bm25_retriever = BM25Retriever.from_documents(processed_chunks)
bm25_retriever.k = 15

ImportError: Could not import rank_bm25, please install with `pip install rank_bm25`.